# Exploratory Data Analysis: Cricket Match Projections

This notebook refactors the original analysis into a production-grade modular structure. It performs the following steps:
1. Downloads and extracts T20 match data from Cricsheet.
2. Parses match metadata (README).
3. Scrapes historical match results from ESPN Cricinfo.
4. Merges metadata with match results for specific team analysis (Bangladesh).
5. Loads and processes detailed ball-by-ball action data.
6. Visualizes strike rates in death overs (17-20) against different opponents.

In [ ]:
import os
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('../src'))
from cricket_processor import CricsheetProcessor, MatchDataProcessor
from espn_scraper import EspnCricinfoScraper

# Constants
TEAM = 'Bangladesh'
FOLDER_NAME = 'Research' # Where the JSON dataset lives
ESPN_URL = 'https://stats.espncricinfo.com/ci/engine/team/25.html?class=3;template=results;type=team;view=results'


## 1. Data Ingestion & Metadata Processing

We use `CricsheetProcessor` to download the zip file and parse the `README.txt` for match IDs and dates.

In [ ]:
cricsheet = CricsheetProcessor(cricsheet_path=FOLDER_NAME)
cricsheet.download_and_extract()

# Process metadata to get the original 'df'
df = cricsheet.process_metadata()

print(f"Processed {len(df)} matches from Cricsheet metadata.")
df.head()

## 2. Scraping Match Results from ESPN Cricinfo

We fetch historical results for Bangladesh using `EspnCricinfoScraper`.

In [ ]:
scraper = EspnCricinfoScraper()
match_results_df = scraper.fetch_team_results(ESPN_URL)

print(f"Fetched {len(match_results_df)} match results from ESPN.")
match_results_df.head()

## 3. Metadata Merging

Merge processed Cricsheet metadata with ESPN results based on the match start date.

In [ ]:
# Ensure 'Start Date' is datetime for merging
match_results_df['Start Date'] = pd.to_datetime(match_results_df['Start Date'], errors='coerce')

# Filter for team-specific matches and merge
team_subset = df[df['Cricsheet_name'].str.contains(TEAM, na=False)][['Cricsheet_id', 'Cricsheet_name', 'date']]

team_df = match_results_df.merge(
    team_subset, 
    left_on='Start Date', 
    right_on='date', 
    how='left'
)

print(f"Merged DataFrame 'team_df' created. Total rows: {len(team_df)}")
num_nans = team_df['Cricsheet_id'].isna().sum()
print(f"Matches without Cricsheet ID matching: {num_nans}")
team_df.head()

## 4. Ball-by-Ball Data Processing

Load detailed CSV data for each match ID found in `team_df`.

In [ ]:
match_proc = MatchDataProcessor(data_folder=FOLDER_NAME)

# Process ball-by-ball for the team
ball_by_ball_df = match_proc.load_ball_by_ball(team_df['Cricsheet_id'], TEAM)

print(f"Loaded {len(ball_by_ball_df)} ball entries for {TEAM}.")
ball_by_ball_df.to_csv('all_new_t20s.csv', index=False)
ball_by_ball_df.head()

## 5. Death Over Analysis (17-20)

Group data to analyze performance in the final overs.

In [ ]:
grouped_data = match_proc.group_by_death_overs(ball_by_ball_df, TEAM, 17, 20)

print(f"Analysis grouping complete. Saving summary to 'bd_17_20.csv'.")
grouped_data.to_csv('bd_17_20.csv', index=False)
grouped_data.head()

## 6. Visualization

Plotting the Strike Rate vs. Bowling Team.

In [ ]:
plt.figure(figsize=(20, 10))

# Create barplot
ax = sns.barplot(x='bowling_team', y='strike_rate', data=grouped_data, errorbar=None, palette='viridis')

plt.title(f'{TEAM} Death Over Strike Rate by Opponent', fontsize=20)
plt.xlabel('Bowling Team', fontsize=16)
plt.ylabel('Strike Rate', fontsize=16)
plt.xticks(rotation=45, ha='right')

# Add labels on top
for i in ax.containers:
    ax.bar_label(i, label_type='edge', fontsize=12, padding=3, fmt='%.1f')

plt.tight_layout()
plt.show()